# Week 3: Feast feature store for IRIS

This notebook demonstrates the mandatory Feast workflow: prepare a timestamped source, apply definitions, materialize the online store, retrieve historical training features, and retrieve online inference features.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import joblib
import pandas as pd
from feast import FeatureStore

ROOT = Path.cwd()
REPO = ROOT / 'feature_repo'
FEATURES = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
FEATURE_REFS = [f'iris_features:{name}' for name in FEATURES]

In [ ]:
subprocess.run(['python', 'src/feast_prepare.py', '--version', 'v1'], check=True)
subprocess.run(['feast', 'apply'], cwd=REPO, check=True)
store = FeatureStore(repo_path=str(REPO))
store.materialize(datetime(2024, 1, 1, tzinfo=timezone.utc), datetime.now(timezone.utc), force=True)
print('Definitions applied and online store materialized.')

In [ ]:
source = pd.read_parquet(REPO / 'data' / 'iris_features.parquet')
entity_df = source[['iris_id', 'event_timestamp', 'species']]
historical = store.get_historical_features(entity_df=entity_df, features=FEATURE_REFS).to_df()
print(historical.head())
print('Historical rows:', len(historical))

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

train, valid = train_test_split(historical, test_size=0.2, stratify=historical['species'], random_state=42)
model = DecisionTreeClassifier(max_depth=3, random_state=1).fit(train[FEATURES], train['species'])
print('Validation accuracy:', accuracy_score(valid['species'], model.predict(valid[FEATURES])))

In [ ]:
ids = [1, 51, 101]
online = store.get_online_features(features=FEATURE_REFS, entity_rows=[{'iris_id': i} for i in ids]).to_dict()
online_df = pd.DataFrame({name: online[name] for name in FEATURES})
pd.DataFrame({'iris_id': ids, 'predicted_species': model.predict(online_df)})